# Taller: POO, Interfaces Gráficas y Bases de Datos en la Nube

En esta lección vamos a crear una aplicación completa utilizando **Python**. Aprenderemos a conectar nuestro programa con **Google Firebase** usando una API, y a crear una interfaz visual con **Tkinter**.

## 1. Definiendo nuestra Entidad (El Backend y POO)
En lugar de usar variables sueltas o listas confusas, vamos a usar Programación Orientada a Objetos para crear un molde (Clase) que representará a cualquier persona que use nuestra aplicación.

In [ ]:
class Usuario:
    def __init__(self, nombre, telefono, edad, comida_favorita):
        self.nombre = nombre
        self.telefono = telefono
        self.edad = edad
        self.comida_favorita = comida_favorita
        
    # Necesitamos convertir nuestro Objeto a un Diccionario
    # porque las APIs en internet (como Firebase) se comunican enviando datos en formato JSON (muy similar a un diccionario en Python).
    def to_dict(self):
        return {
            "nombre": self.nombre,
            "telefono": self.telefono,
            "edad": self.edad,
            "comida_favorita": self.comida_favorita
        }

## 2. La Lógica de Conexión a Firebase (Servicios)

Firebase proporciona un servicio llamado **Realtime Database**. Para usarlo en su forma más sencilla, solo necesitamos hacer peticiones HTTP (como las que hace nuestro navegador web) mediante la librería externa `requests` de Python.

Vamos a crear una clase `GestorUsuarios` que se encargará de:
1. **Descargar** toda la base de datos de Firebase cada vez que inicia el programa.
2. **Sobrescribir** la base de datos de Firebase cada vez que cerramos el programa.

In [ ]:
import requests # <- Asegúrate de tener instalado requests en tu entorno (pip install requests)

# IMPORTANTE: Para que esto funcione, debes crear un proyecto en Firebase,
# generar una Realtime Database y poner sus reglas en 'Modo de Prueba' (true autorizadas).
# Cambia la URL de aquí abajo por la tuya. Termina en '.json' por exigencia de la API de Firebase.
FIREBASE_URL = "https://TU-PROYECTO-DEFAULT-rtdb.firebaseio.com/usuarios.json"

class GestorUsuarios:
    def __init__(self):
        self.lista_usuarios = []
        
    def agregar_localmente(self, usuario):
        self.lista_usuarios.append(usuario)

    def cargar_desde_firebase(self):
        print("Conectando a Firebase para descargar datos...")
        try:
            # Un 'GET' request lee la información de la URL
            respuesta = requests.get(FIREBASE_URL)
            
            if respuesta.status_code == 200 and respuesta.json() is not None:
                datos = respuesta.json()
                self.lista_usuarios = []
                
                # Firebase nos devuelve un diccionario de diccionarios.
                # Lo convertimos nuevamente en Objetos 'Usuario' de nuestra propia clase
                for key, val in datos.items():
                    obj_usuario = Usuario(val['nombre'], val['telefono'], val['edad'], val['comida_favorita'])
                    self.lista_usuarios.append(obj_usuario)
                    
                print(f"¡Éxito! Se cargaron {len(self.lista_usuarios)} usuarios de la nube al iniciar el programa.")
            else:
                print("La base de datos en nube está vacía o recién creada.")
        except Exception as e:
            print("Error al intentar cargar datos de Firebase:", e)

    def guardar_en_firebase(self):
        """Ésta función se llamará al cerrar nuestra ventana visual"""
        print("Subiendo todos los datos locales a Firebase...")
        
        # Transformamos toda nuestra lista de objetos a un gran diccionario
        # Porque Firebase (JSON) no entiende directamente de Objetos en Python.
        diccionario_a_subir = {}
        for i, usuario in enumerate(self.lista_usuarios):
            diccionario_a_subir[f"usuario_{i}"] = usuario.to_dict()
            
        try:
            # Un 'PUT' request reemplaza TODO el contenido actual en esa URL con nuestra información limpia
            respuesta = requests.put(FIREBASE_URL, json=diccionario_a_subir)
            if respuesta.status_code == 200:
                print("¡Datos respaldados exitosamente de regreso en la Nube!")
            else:
                print("Error del servidor al subir:", respuesta.status_code)
        except Exception as e:
            print("No se pudo guardar la información a Firebase:", e)

## 3. La Interfaz Gráfica (El Frontend con Tkinter)
Tkinter es una biblioteca nativa de Python útil para generar ventanas en nuestro sistema operativo. Aquí veremos cómo enlazar los botones para que usen nuestra lógica de Backend, y específicamente, **cómo capturar el momento de cerrar la ventana (la 'X') para ejecutar el guardado total hacia Firebase.**

In [ ]:
import tkinter as tk
from tkinter import messagebox

class InterfazAplicacion:
    def __init__(self, ventana, gestor):
        self.ventana = ventana
        self.gestor = gestor
        self.ventana.title("Gestión de Perfiles")
        
        # ----- PASO 1. AL INICIAR: CARGAR ----- 
        # Esto sucede automáticamente en cuanto se dibuja esta ventana
        self.gestor.cargar_desde_firebase()
        
        # ----- PASO 2. AL CERRAR: GUARDAR ----- 
        # Capturamos el evento 'WM_DELETE_WINDOW' que ocurre cuando presionas el botón 'X'
        self.ventana.protocol("WM_DELETE_WINDOW", self.evento_cerrar_ventana)
        
        # ----- DISEÑO DE LA APP (GRID SYSTEM EN TKINTER) -----
        tk.Label(ventana, text="Nombre:").grid(row=0, column=0, padx=10, pady=5)
        self.caja_nombre = tk.Entry(ventana)
        self.caja_nombre.grid(row=0, column=1, padx=10, pady=5)
        
        tk.Label(ventana, text="Teléfono:").grid(row=1, column=0, padx=10, pady=5)
        self.caja_tel = tk.Entry(ventana)
        self.caja_tel.grid(row=1, column=1, padx=10, pady=5)
        
        tk.Label(ventana, text="Edad:").grid(row=2, column=0, padx=10, pady=5)
        self.caja_edad = tk.Entry(ventana)
        self.caja_edad.grid(row=2, column=1, padx=10, pady=5)
        
        tk.Label(ventana, text="Comida Favorita:").grid(row=3, column=0, padx=10, pady=5)
        self.caja_comida = tk.Entry(ventana)
        self.caja_comida.grid(row=3, column=1, padx=10, pady=5)
        
        # Botones
        btn_agregar = tk.Button(ventana, text="Guardar Usuario (En Memoria Local)", command=self.agregar)
        btn_agregar.grid(row=4, column=0, columnspan=2, pady=10)
        
        btn_ver = tk.Button(ventana, text="Ver Lista (Ver Log en Consola)", command=self.mostrar)
        btn_ver.grid(row=5, column=0, columnspan=2, pady=5)

    def agregar(self):
        # Extraer texto de las cajas de GUI
        nom = self.caja_nombre.get()
        tel = self.caja_tel.get()
        ed = self.caja_edad.get()
        com = self.caja_comida.get()
        
        # Validar que no estén vacíos
        if nom and tel and ed and com:
            # Instanciamos la clase y al agregamos al local backend
            nuevo = Usuario(nom, tel, ed, com)
            self.gestor.agregar_localmente(nuevo)
            messagebox.showinfo("Aviso", "El usuario se agregó a la memoria con éxito.")
            
            # Limpiar campos
            self.caja_nombre.delete(0, tk.END)
            self.caja_tel.delete(0, tk.END)
            self.caja_edad.delete(0, tk.END)
            self.caja_comida.delete(0, tk.END)
        else:
            messagebox.showwarning("Error", "Debe llenar todos los campos antes de guardar.")
            
    def mostrar(self):
        print("\n--- Usuarios Actuales en Memoria ---")
        for u in self.gestor.lista_usuarios:
            print(f"{u.nombre} - Tel: {u.telefono} - Edad: {u.edad} - Comida: {u.comida_favorita}")
        print("------------------------------------\n")

    def evento_cerrar_ventana(self):
        # Esta es la magia que conecta el cierre de programa con el ciclo de vida del guardado total.
        if messagebox.askokcancel("Salir de la App", "¿Deseas cerrar y guardar toda la info recién agregada hacia Firebase?"):
            # 1. Ejecutamos el guardado de la clase Backend
            self.gestor.guardar_en_firebase()
            # 2. Por último, destruimos el frontend para detener la ejecución y el ciclo mainloop
            self.ventana.destroy()

## 4. Juntando Todo (Main)
Ejecutando esta última celda levantaremos el **Gestor** (backend) y la asignaremos a nuestra ventana **Tkinter** (frontend). Al terminar, cuando los alumnos decidan cerrar la ventana visual en la 'X', verán como los datos son enviados remotamente.  

In [ ]:
if __name__ == '__main__':
    # 1. Crear el cerebro de datos (Backend/Modelo)
    gestor_principal = GestorUsuarios()
    
    # 2. Crear el Contenedor Visual (Ventana Raíz GUI)
    ventana_raiz = tk.Tk()
    
    # 3. Lanzar la aplicación (pasándole la ventana para que se dibuje y el backend)
    app = InterfazAplicacion(ventana_raiz, gestor_principal)
    
    # 4. Iniciar el Loop incesante, deteniendo la ejecución hasta cerrar la ventana
    print("\n=> Aplicación Gráfica Activa. Cierra la ventana pop-up (con la X) para detener y ejecutar el subido a nube.\n")
    ventana_raiz.mainloop()